# 1. Data Analysis
Comprehensive exploratory analysis of the HR analytics dataset used in this project.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

TRAIN_PATH = ROOT / 'src/data/train.csv'
TEST_PATH = ROOT / 'src/data/test.csv'


In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print('Train shape:', train.shape)
print('Test shape :', test.shape)
print('Columns    :', list(train.columns))
train.head()


In [ ]:
summary = pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'missing_pct': (train.isna().mean() * 100).round(2),
    'n_unique': train.nunique(dropna=False)
}).sort_values('missing_pct', ascending=False)
summary


In [ ]:
target_dist = train['target'].value_counts().sort_index()
target_rate = train['target'].mean()

print('Target counts:')
print(target_dist)
print(f'Positive class rate: {target_rate:.3f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=train, x='target', ax=ax[0])
ax[0].set_title('Target Distribution')

ax[1].pie(target_dist.values, labels=target_dist.index, autopct='%1.1f%%', startangle=90)
ax[1].set_title('Target Proportion')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train['training_hours'], bins=40, kde=True, ax=axes[0])
axes[0].set_title('Training Hours Distribution')

sns.histplot(train['city_development_index'], bins=30, kde=True, ax=axes[1], color='orange')
axes[1].set_title('City Development Index Distribution')

plt.tight_layout()
plt.show()


In [ ]:
categorical_cols = [c for c in train.columns if train[c].dtype == 'object']
cardinality = train[categorical_cols].nunique(dropna=False).sort_values(ascending=False)
print('Categorical feature cardinality:')
display(cardinality.to_frame('n_unique'))

print('\nTop cities by frequency:')
display(train['city'].value_counts().head(10).to_frame('count'))


In [ ]:
group_cols = ['relevent_experience', 'education_level', 'enrolled_university']
for col in group_cols:
    target_by_col = train.groupby(col, dropna=False)['target'].mean().sort_values(ascending=False)
    print(f'\nMean target by {col}:')
    display(target_by_col.to_frame('mean_target'))


In [ ]:
numeric_cols = ['city_development_index', 'training_hours', 'target']
corr = train[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='Blues')
plt.title('Numeric Correlation Matrix')
plt.show()


In [ ]:
profile = pd.DataFrame({
    'train_missing_pct': (train.isna().mean() * 100).round(2),
    'test_missing_pct': (test.isna().mean() * 100).round(2),
    'train_unique': train.nunique(dropna=False),
    'test_unique': test.nunique(dropna=False),
})
profile.sort_values('train_missing_pct', ascending=False)
